# Topological Illustration

This notebook rebuilds the topology figure from a deterministic point-cloud illustration and the released MetaMatch topological statistics.

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
FIGURES = ROOT / 'figures'
META = FIGURES / 'topology_vr_h0_h1_h2_metamatch_metadata.json'

with open(META) as f:
    metadata = json.load(f)

plt.rcParams.update({
    'figure.dpi': 130,
    'savefig.dpi': 300,
    'font.size': 9,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

def save_fig(fig, name):
    FIGURES.mkdir(exist_ok=True)
    fig.savefig(FIGURES / f'{name}.png', bbox_inches='tight')
    fig.savefig(FIGURES / f'{name}.pdf', bbox_inches='tight')

## Helper Functions Used to Draw Vietoris-Rips Complexes

In [ ]:
def make_cloud(seed=0, n=42, radius=1.0, noise=0.08, offset=(0.0, 0.0)):
    rng = np.random.default_rng(seed)
    theta = np.linspace(0, 2*np.pi, n, endpoint=False)
    r = radius + rng.normal(0, noise, n)
    x = r * np.cos(theta) + rng.normal(0, noise, n) + offset[0]
    y = r * np.sin(theta) + rng.normal(0, noise, n) + offset[1]
    return np.c_[x, y]

def pairwise_dist(points):
    diff = points[:, None, :] - points[None, :, :]
    return np.sqrt((diff ** 2).sum(axis=2))

def vr_edges(points, eps):
    d = pairwise_dist(points)
    idx = np.transpose(np.triu(d <= eps, k=1).nonzero())
    return [(points[i], points[j]) for i, j in idx]

def draw_vr(ax, points, eps, color='#2CA02C'):
    edges = vr_edges(points, eps)
    if edges:
        ax.add_collection(LineCollection(edges, colors=color, linewidths=1.0, alpha=0.35))
    ax.scatter(points[:, 0], points[:, 1], s=14, color='#222222', zorder=3)
    ax.set_aspect('equal')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlim(points[:, 0].min() - .25, points[:, 0].max() + .25)
    ax.set_ylim(points[:, 1].min() - .25, points[:, 1].max() + .25)

def draw_diagram(ax, stats, prefix, color):
    h0 = stats.get(f'tda_h0_max_life_{prefix}', 0.0)
    h1 = stats.get(f'tda_h1_max_life_{prefix}', 0.0)
    h2 = stats.get(f'tda_h2_max_life_{prefix}', 0.0)
    xs = [0.02, 0.35, 0.70]
    ys = [h0, h1, h2]
    labels = ['H0', 'H1', 'H2']
    ymax = max(max(ys) * 1.15, 1.0)
    ax.plot([0, ymax], [0, ymax], color='#777777', linewidth=0.9)
    ax.scatter(xs, ys, s=[30, 45, 55], c=color, edgecolor='white', linewidth=0.8)
    for x, y, lab in zip(xs, ys, labels):
        ax.text(x + 0.03, y, lab, va='center', fontsize=8)
    ax.set_xlim(0, ymax)
    ax.set_ylim(0, ymax)
    ax.set_xlabel('Birth')
    ax.set_ylabel('Death')

## Build the Figure

In [ ]:
match_row, non_match_row = metadata['rows']
rows = [('Matching pair', match_row, make_cloud(seed=3, offset=(-0.18, 0)), make_cloud(seed=4, offset=(0.18, 0))),
        ('Non-matching pair', non_match_row, make_cloud(seed=3, offset=(-0.65, 0)), make_cloud(seed=11, offset=(0.65, 0)))]

fig, axes = plt.subplots(2, 6, figsize=(13, 5.2), gridspec_kw={'width_ratios': [1, 1, 1, 1, 1.1, 1.1]})
for r, (label, row, src, tgt) in enumerate(rows):
    points = np.vstack([src, tgt])
    eps_values = row['eps_values']
    for c, eps in enumerate(eps_values):
        draw_vr(axes[r, c], points, eps)
        axes[r, c].set_title(f'eps={eps:.2f}')
    stats = row['used_real_metamatch_tda_values']
    draw_diagram(axes[r, 4], stats, 'src', '#1F77B4')
    axes[r, 4].set_title('Source')
    draw_diagram(axes[r, 5], stats, 'combined', '#D62728')
    axes[r, 5].set_title('Combined')
    axes[r, 0].set_ylabel(label, fontsize=10)

fig.tight_layout()
save_fig(fig, 'topology_vr_h0_h1_h2_metamatch')
plt.show()

## Real Topological Values Used in the Figure

In [ ]:
for row in metadata['rows']:
    print(row['source_column'], '->', row['target_column'], '| label=', row['label'])
    vals = row['used_real_metamatch_tda_values']
    keep = {k: vals[k] for k in vals if k in {
        'tda_h0_max_life_combined', 'tda_h1_max_life_combined', 'tda_h2_max_life_combined',
        'tda_h0_entropy_combined', 'tda_h1_entropy_combined', 'tda_h2_entropy_combined',
        'tda_h0_bottleneck', 'tda_h0_wasserstein', 'tda_h1_bottleneck', 'tda_h1_wasserstein'
    }}
    display(keep)